# Pertemuan 5 — Metode Pemulusan Eksponensial
## Notebook Praktikum · Analisis Runtun Waktu (SDT-3329)

**Program Studi Sains Data · Fakultas Digital, Desain dan Bisnis · Institut Teknologi Sains Bandung**
Semester 6 · 3 SKS (T: 2, P: 1) · Mata kuliah syarat: Statistika Komputasi, Pemrograman Statistika, Aljabar Linear ·
Dosen pengampu: Bakti Siregar, S.Si., M.Sc

---

### Keselarasan dengan RPS (OBE)

| Komponen | Isi |
|---|---|
| **Kemampuan akhir (Sub-CPMK)** | Mahasiswa mampu menerapkan metode pemulusan eksponensial (Single, Double, Triple Exponential Smoothing) untuk peramalan. |
| **CPL / CPMK** | CPL-1 · CPL-2 · CPMK-3 |
| **Indikator** | Mampu memilih metode pemulusan eksponensial yang tepat dan menerapkannya untuk peramalan data runtun waktu. |
| **Kriteria penilaian** | Ketepatan pemilihan metode dan akurasi hasil peramalan. |
| **Materi pembelajaran** | SES, Holt (double), Holt-Winters (triple), pemilihan parameter pemulusan. |
| **Konsep pemrograman** | Algoritma rekursif ber-*state*; fungsi objektif sebagai objek untuk `scipy.optimize`; kelas peramal (bagian 🧠) |
| **Bobot penilaian minggu ini** | 4% |
| **Alokasi waktu** | TM 2 × 50′ · Praktikum 1 × 170′ · PT 2 × 60′ · BM 2 × 60′ |
| **Keterkaitan tugas (RTM)** | — (bekal Tugas 2: pemulusan eksponensial) |
| **Pustaka** | Hyndman & Athanasopoulos (2021) Bab 8 |

**Dataset:** Debit Sungai Nil tahunan 1871–1970 · PDB riil AS (rata-rata tahunan) · Penjualan mobil bulanan Quebec 1960–1968
**Pustaka Python:** pandas · NumPy · Matplotlib · seaborn · statsmodels · SciPy

### Peta notebook

| Bagian | Isi | Indikator RPS |
|---|---|---|
| 1 | Gagasan pemulusan: dari rata-rata dan naive ke bobot eksponensial | — |
| 2 | **SES** dari nol: rekursi, bobot, memilih α dengan optimasi | Menerapkan pemulusan eksponensial |
| 3 | **Holt** (tren) dan Holt *damped* | Memilih metode yang tepat |
| 4 | **Holt-Winters** aditif vs multiplikatif — implementasi sendiri sebagai kelas | Menerapkan untuk peramalan |
| 5 | Pemilihan metode dan parameter dengan data uji kronologis | **Akurasi hasil peramalan** |
| 6 | Interval prediksi dengan model ETS (state-space) | Akurasi hasil peramalan |
| 7 | 🧠 Konsep pemrograman: algoritma rekursif ber-*state*, fungsi objektif sebagai objek, kelas peramal | — |
| 8 | Latihan, ringkasan, logbook | — |

### Cara menggunakan notebook ini

- Jalankan sel berurutan (Google Colab atau Jupyter lokal). Di Colab, `statsmodels` sudah tersedia; `arch` (P11) dipasang dengan `pip install arch`.
- Sel **LATIHAN** berisi `TODO`; lengkapi selama praktikum. Bagian 🧠 **Konsep Pemrograman** menjelaskan prinsip di balik kode,
  bukan sekadar cara memanggil pustaka.
- Penilaian memakai rubrik RPS — *Kualitas Implementasi (Kode & Model)* dan *Interpretasi & Visualisasi Hasil* — serta
  Rubrik PRAKTIKUM (eksekusi 40%, pembersihan data 35%, penanganan error & modularitas 25%).
- Unggah notebook dan logbook ke LMS/Virtual Lab setelah praktikum.

---
## 0. Persiapan

In [29]:
# 1. Import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import statsmodels.api as sm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9, 'figure.figsize': (10, 3.4)})
pd.set_option('display.width', 140)
RANDOM_STATE = 42

# 2. Import metode Exponential Smoothing dan optimasi
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from scipy.optimize import minimize_scalar, minimize

# 3. Dataset Nile: debit sungai bulanan (data aktual)
from pangaeapy import PanDataSet

ds = PanDataSet(218345)
nil_raw = ds.data.copy()

nil = pd.Series(
    pd.to_numeric(nil_raw.iloc[:, 1], errors='coerce').values,
    index=pd.to_datetime(nil_raw.iloc[:, 0], errors='coerce'),
    name='Nile Discharge'
).dropna().sort_index()

nil.index = nil.index.to_period('M').to_timestamp()

# 4. Dataset GDP: Real GDP Amerika Serikat (data aktual kuartalan)
url_gdp = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=GDPC1'

gdp = pd.read_csv(url_gdp, parse_dates=['observation_date'])
gdp = gdp.set_index('observation_date')['GDPC1']
gdp = pd.to_numeric(gdp, errors='coerce').dropna()
gdp.name = 'Real GDP'

# 5. Dataset Car Sales: penjualan mobil bulanan
url_car = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-car-sales.csv'

car = pd.read_csv(
    url_car,
    parse_dates=['Month'],
    index_col='Month'
)['Sales'].asfreq('MS')

car.name = 'Car Sales'

# 6. Menyamakan rentang periode berdasarkan Car Sales
start, end = car.index.min(), car.index.max()

nil = nil.loc[start:end]
gdp = gdp.loc[start:end]

# 7. Fungsi evaluasi akurasi forecasting
def mae(a, f):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(f))))

def rmse(a, f):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(f)) ** 2)))

def mape(a, f):
    return float(100 * np.mean(np.abs((np.asarray(a) - np.asarray(f)) / np.asarray(a))))

# 8. Informasi dataset
print(f"Periode: {start:%Y-%m} sampai {end:%Y-%m}")
print(f"Nile      : {len(nil)} observasi (Bulanan)")
print(f"GDP       : {len(gdp)} observasi (Kuartalan)")
print(f"Car Sales : {len(car)} observasi (Bulanan)")

Periode: 1960-01 sampai 1968-12
Nile      : 108 observasi (Bulanan)
GDP       : 36 observasi (Kuartalan)
Car Sales : 108 observasi (Bulanan)


## 1. Gagasan Pemulusan (Smoothing)

Perhatikan data series berikut:

In [30]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=['Nile River Discharge', 'Real GDP', 'Monthly Car Sales']
)

for i, (data, name) in enumerate([
    (nil, 'Nile'),
    (gdp, 'GDP'),
    (car, 'Car Sales')
], 1):
    fig.add_trace(
        go.Scatter(x=data.index, y=data, mode='lines', name=name),
        row=i, col=1
    )

fig.update_layout(
    title='Time Series Datasets',
    height=700,
    template='plotly_white',
    showlegend=False,
    hovermode='x unified'
)

fig.show()

Dalam Time Series Analysis, terdapat tiga pendekatan sederhana untuk meramalkan nilai periode berikutnya:

1. **Naive:** Menggunakan hanya data aktual terakhir sebagai ramalan.

   $$\hat{y}_{T+1}=y_T$$

2. **Rata-rata (Average):** Menggunakan seluruh data historis dengan bobot yang sama.

   $$\hat{y}_{T+1}=\bar{y}=\frac{1}{T}\sum_{t=1}^{T}y_t$$

3. **Exponential Smoothing:** Menggunakan seluruh data historis, tetapi memberikan bobot lebih besar pada data terbaru. Semakin lama usia data, semakin kecil bobotnya.

   $$\hat{y}_{T+1}=
   \alpha y_T+
   \alpha(1-\alpha)y_{T-1}+
   \alpha(1-\alpha)^2y_{T-2}+\cdots$$

   dengan $0<\alpha\leq1$.

**Keterangan:** $\alpha$ (*smoothing parameter*) mengatur pengaruh data terbaru terhadap ramalan. Alpha besar membuat ramalan lebih responsif, sedangkan Alpha kecil menghasilkan pemulusan yang lebih halus.

**Intinya:** Naive hanya menggunakan data terakhir, Average memberikan bobot sama, sedangkan Exponential Smoothing memberikan bobot lebih besar pada data terbaru.

In [31]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import SimpleExpSmoothing


# ============================================================
# 2. PARAMETERS
# ============================================================

FORECAST_YEARS = 2
ALPHA = 0.3

datasets = [
    (nil, 'Nile River Discharge'),
    (gdp, 'United States Real GDP'),
    (car, 'Monthly Car Sales')
]


# ============================================================
# 3. FORECAST FUNCTION
# ============================================================

def forecast_comparison(data, forecast_years=2, alpha=0.3):

    data = data.dropna().sort_index()

    # Detect frequency
    freq = pd.infer_freq(data.index)

    if freq is None:
        raise ValueError('Frekuensi data tidak teratur.')

    # Forecast horizon
    if freq.upper().startswith('Q'):
        horizon = forecast_years * 4

    elif freq.upper().startswith('M'):
        horizon = forecast_years * 12

    elif freq.upper().startswith(('Y', 'A')):
        horizon = forecast_years

    else:
        raise ValueError(f'Frekuensi tidak dikenali: {freq}')

    # Future dates
    future_index = pd.date_range(
        start=data.index[-1],
        periods=horizon + 1,
        freq=freq
    )[1:]


    # --------------------------------------------------------
    # NAIVE
    # --------------------------------------------------------

    naive_smoothing = data.shift(1)

    naive_forecast = np.repeat(
        data.iloc[-1], horizon
    )


    # --------------------------------------------------------
    # AVERAGE
    # --------------------------------------------------------

    average_smoothing = (
        data.expanding().mean().shift(1)
    )

    average_forecast = np.repeat(
        data.mean(), horizon
    )


    # --------------------------------------------------------
    # SIMPLE EXPONENTIAL SMOOTHING (SES)
    # --------------------------------------------------------

    model = SimpleExpSmoothing(
        data,
        initialization_method='estimated'
    )

    fitted_model = model.fit(
        smoothing_level=alpha,
        optimized=False
    )

    ses_smoothing = fitted_model.fittedvalues

    ses_forecast = fitted_model.forecast(horizon)


    # --------------------------------------------------------
    # COMBINE RESULTS
    # --------------------------------------------------------

    smoothing = pd.DataFrame({
        'Naive': naive_smoothing,
        'Average': average_smoothing,
        'SES': ses_smoothing
    }, index=data.index)

    forecast = pd.DataFrame({
        'Naive': naive_forecast,
        'Average': average_forecast,
        'SES': ses_forecast.values
    }, index=future_index)

    return smoothing, forecast


# ============================================================
# 4. COLORS
# ============================================================

colors = {
    'Actual': '#333333',
    'Naive': '#1f77b4',
    'Average': '#ff7f0e',
    'SES': '#2ca02c'
}


# ============================================================
# 5. CREATE SUBPLOTS
# ============================================================

fig = make_subplots(
    rows=3,
    cols=1,
    subplot_titles=[name for _, name in datasets],
    vertical_spacing=0.12
)

forecast_results = {}


# ============================================================
# 6. PLOT ACTUAL, SMOOTHING AND FORECAST
# ============================================================

for i, (data, name) in enumerate(datasets, start=1):

    data = data.dropna().sort_index()

    smoothing, forecast = forecast_comparison(
        data,
        forecast_years=FORECAST_YEARS,
        alpha=ALPHA
    )

    forecast_results[name] = forecast

    # Actual data
    fig.add_trace(
        go.Scatter(
            x=data.index,
            y=data.values,
            name='Actual',
            legendgroup='Actual',
            showlegend=(i == 1),
            line=dict(color=colors['Actual'], width=1.8)
        ),
        row=i, col=1
    )

    # Naive smoothing
    fig.add_trace(
        go.Scatter(
            x=smoothing.index,
            y=smoothing['Naive'],
            name='Naive - Smoothing',
            legendgroup='Naive',
            showlegend=(i == 1),
            line=dict(color=colors['Naive'], width=2)
        ),
        row=i, col=1
    )

    # Naive forecast
    fig.add_trace(
        go.Scatter(
            x=forecast.index,
            y=forecast['Naive'],
            name='Naive - Forecast',
            legendgroup='Naive',
            showlegend=(i == 1),
            line=dict(color=colors['Naive'], dash='dash', width=2.5),
            mode='lines+markers'
        ),
        row=i, col=1
    )

    # Average smoothing
    fig.add_trace(
        go.Scatter(
            x=smoothing.index,
            y=smoothing['Average'],
            name='Average - Smoothing',
            legendgroup='Average',
            showlegend=(i == 1),
            line=dict(color=colors['Average'], width=2)
        ),
        row=i, col=1
    )

    # Average forecast
    fig.add_trace(
        go.Scatter(
            x=forecast.index,
            y=forecast['Average'],
            name='Average - Forecast',
            legendgroup='Average',
            showlegend=(i == 1),
            line=dict(color=colors['Average'], dash='dash', width=2.5),
            mode='lines+markers'
        ),
        row=i, col=1
    )

    # SES smoothing
    fig.add_trace(
        go.Scatter(
            x=smoothing.index,
            y=smoothing['SES'],
            name=f'SES - Smoothing (α={ALPHA})',
            legendgroup='SES',
            showlegend=(i == 1),
            line=dict(color=colors['SES'], width=2.5)
        ),
        row=i, col=1
    )

    # SES forecast
    fig.add_trace(
        go.Scatter(
            x=forecast.index,
            y=forecast['SES'],
            name='SES - Forecast',
            legendgroup='SES',
            showlegend=(i == 1),
            line=dict(color=colors['SES'], dash='dash', width=2.5),
            mode='lines+markers'
        ),
        row=i, col=1
    )

    # Forecast boundary
    fig.add_vline(
        x=data.index[-1],
        line_dash='dot',
        line_width=1,
        line_color='gray',
        row=i, col=1
    )


# ============================================================
# 7. UPDATE LAYOUT
# ============================================================

fig.update_layout(
    title=dict(
        text='<b>Naive, Average and Simple Exponential Smoothing Forecast</b>',
        x=0.5,
        xanchor='center'
    ),
    height=1100,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        orientation='h',
        y=-0.10,
        x=0.5,
        xanchor='center'
    ),
    margin=dict(l=80, r=40, t=100, b=160)
)


# ============================================================
# 8. AXIS TITLES
# ============================================================

fig.update_xaxes(title_text='Year', showgrid=True)
fig.update_yaxes(title_text='Observed Value', showgrid=True)


# ============================================================
# 9. DISPLAY VISUALIZATION
# ============================================================

fig.show()

**Catatan:** SES (Simple Exponential Smoothing) lebih sesuai digunakan ketika data memiliki pola relatif stabil, tanpa tren dan tanpa pola musiman yang signifikan. Keunggulan SES adalah kemampuannya memberikan bobot lebih besar pada data terbaru melalui parameter $\alpha$, sehingga ramalan dapat menyesuaikan perubahan level data secara bertahap.

---
## 2. Simple Exponential Smoothing (SES)

Bentuk rekursif (bentuk komponen): $\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}$, ramalan $\hat y_{T+h} = \ell_T$ untuk semua $h$
— ramalan **datar**. Cocok untuk deret **tanpa tren dan tanpa musiman**.

In [32]:

# ============================================================
# 1. PARAMETERS
# ============================================================

FORECAST_YEARS = 2

ALPHAS = [0.1, 0.3, 0.5, 0.7, 0.9]

datasets = [
    (nil, 'Nile River Discharge'),
    (gdp, 'United States Real GDP'),
    (car, 'Monthly Car Sales')
]


# ============================================================
# 2. SES SIMULATION
# ============================================================

def compare_alpha(data, alphas, forecast_years=2):

    data = data.dropna().sort_index()

    # Detect frequency
    freq = pd.infer_freq(data.index)

    if freq is None:
        raise ValueError(
            f'Frekuensi data tidak teratur: {data.name}'
        )

    # Determine forecast horizon
    if freq.upper().startswith('Q'):
        horizon = forecast_years * 4

    elif freq.upper().startswith('M'):
        horizon = forecast_years * 12

    elif freq.upper().startswith(('Y', 'A')):
        horizon = forecast_years

    else:
        raise ValueError(
            f'Frekuensi tidak dikenali: {freq}'
        )

    # Generate future dates
    future_index = pd.date_range(
        start=data.index[-1],
        periods=horizon + 1,
        freq=freq
    )[1:]

    fig = go.Figure()

    # --------------------------------------------------------
    # ACTUAL DATA
    # --------------------------------------------------------

    fig.add_trace(
        go.Scatter(
            x=data.index,
            y=data.values,
            name='Actual',
            line=dict(color='black', width=2)
        )
    )

    # Store forecast results
    results = {}

    # Store smoothing results
    smoothing_results = {}

    # --------------------------------------------------------
    # SES FOR EACH ALPHA
    # --------------------------------------------------------

    for alpha in alphas:

        model = SimpleExpSmoothing(
            data,
            initialization_method='estimated'
        )

        fitted = model.fit(
            smoothing_level=alpha,
            optimized=False
        )

        # In-sample smoothing
        smoothing = fitted.fittedvalues

        # Out-of-sample forecast
        forecast = fitted.forecast(horizon)

        results[alpha] = forecast.values
        smoothing_results[alpha] = smoothing.values

        # Plot smoothing
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=smoothing,
                name=f'SES Smoothing (α={alpha})',
                line=dict(width=2)
            )
        )

        # Plot forecast
        fig.add_trace(
            go.Scatter(
                x=future_index,
                y=forecast.values,
                name=f'SES Forecast (α={alpha})',
                line=dict(width=2.5, dash='dash'),
                mode='lines+markers'
            )
        )

    # --------------------------------------------------------
    # FORECAST BOUNDARY
    # Fix Plotly Timestamp compatibility
    # --------------------------------------------------------

    boundary_date = data.index[-1].strftime('%Y-%m-%d')

    fig.add_shape(
        type='line',
        x0=boundary_date,
        x1=boundary_date,
        y0=0,
        y1=1,
        xref='x',
        yref='paper',
        line=dict(
            color='gray',
            width=1.5,
            dash='dot'
        )
    )

    fig.add_annotation(
        x=boundary_date,
        y=1,
        xref='x',
        yref='paper',
        text='Forecast Start',
        showarrow=False,
        yshift=12,
        font=dict(size=10, color='gray')
    )

    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    fig.update_layout(
        title=dict(
            text=(
                f'<b>SES Forecast Simulation: '
                f'{data.name if data.name else "Time Series"}</b>'
            ),
            x=0.5,
            xanchor='center'
        ),
        xaxis_title='Time',
        yaxis_title='Observed Value',
        template='plotly_white',
        hovermode='x unified',
        height=650,
        legend=dict(
            orientation='h',
            y=-0.2,
            x=0.5,
            xanchor='center'
        ),
        margin=dict(
            l=70,
            r=40,
            t=100,
            b=150
        )
    )

    fig.show()

    # --------------------------------------------------------
    # RETURN FORECAST TABLE
    # --------------------------------------------------------

    forecast_table = pd.DataFrame(
        results,
        index=future_index
    )

    forecast_table.columns = [
        f'Alpha {alpha}' for alpha in alphas
    ]

    return forecast_table, smoothing_results


# ============================================================
# 3. RUN SIMULATION
# ============================================================

forecast_results = {}
smoothing_results = {}

for data, name in datasets:

    print('\n' + '=' * 65)
    print(name)
    print('=' * 65)

    try:

        forecast_table, smoothing_table = compare_alpha(
            data,
            ALPHAS,
            FORECAST_YEARS
        )

        forecast_results[name] = forecast_table
        smoothing_results[name] = smoothing_table

        print('\nFORECAST RESULTS')

        print(
            forecast_table.round(2).to_string()
        )

    except Exception as e:

        print(f'Error pada {name}: {e}')



Nile River Discharge



FORECAST RESULTS
            Alpha 0.1  Alpha 0.3  Alpha 0.5  Alpha 0.7  Alpha 0.9
1969-01-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-02-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-03-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-04-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-05-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-06-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-07-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-08-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-09-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-10-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-11-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1969-12-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1970-01-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,227.1700
1970-02-01 1,905.2000 1,536.1100 1,367.1700 1,275.2200 1,2


FORECAST RESULTS
            Alpha 0.1  Alpha 0.3  Alpha 0.5  Alpha 0.7  Alpha 0.9
1969-01-01 4,745.1100 5,093.6200 5,162.9600 5,189.2900 5,199.6800
1969-04-01 4,745.1100 5,093.6200 5,162.9600 5,189.2900 5,199.6800
1969-07-01 4,745.1100 5,093.6200 5,162.9600 5,189.2900 5,199.6800
1969-10-01 4,745.1100 5,093.6200 5,162.9600 5,189.2900 5,199.6800
1970-01-01 4,745.1100 5,093.6200 5,162.9600 5,189.2900 5,199.6800
1970-04-01 4,745.1100 5,093.6200 5,162.9600 5,189.2900 5,199.6800
1970-07-01 4,745.1100 5,093.6200 5,162.9600 5,189.2900 5,199.6800
1970-10-01 4,745.1100 5,093.6200 5,162.9600 5,189.2900 5,199.6800

Monthly Car Sales



FORECAST RESULTS
             Alpha 0.1   Alpha 0.3   Alpha 0.5   Alpha 0.7   Alpha 0.9
1969-01-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-02-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-03-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-04-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-05-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-06-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-07-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-08-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-09-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-10-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-11-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1969-12-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 14,872.2100
1970-01-01 17,554.0300 17,133.4200 16,297.0800 15,569.7100 

**Interpretasi hasil:**

Ketika menjalankan kode tersebut, perhatikan:
- α = 0.1: garis smoothing relatif halus dan tidak terlalu mengikuti fluktuasi data.
- α = 0.3: garis smoothing mulai merespons perubahan data.
- α = 0.5: respons terhadap data aktual semakin besar.
- α = 0.7: garis smoothing lebih dekat dengan data aktual.
- α = 0.9: garis smoothing sangat responsif dan cenderung mengikuti perubahan data terbaru.

Selanjutnya, perhatikan proses melakukan Simple Exponential Smoothing (SES) sebagai berikut:

### Persiapan awal: Import library

In [33]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.optimize import minimize_scalar
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

pd.set_option("display.float_format", "{:,.4f}".format)

### Step 1. Persiapan data

Tujuan: Menyiapkan data time series yang akan digunakan dalam perhitungan SES. Pada contoh ini kita menggunakan dataset nil yang sudah tersedia.

In [34]:
# Mengambil dan membersihkan data
y = pd.Series(nil).dropna().astype(float)

# Menentukan initial level
L0 = y.iloc[0]

# Menampilkan informasi data
print("INFORMASI DATA")
print("-" * 45)

print(f"Jumlah observasi : {len(y)}")
print(f"Periode awal     : {y.index[0]}")
print(f"Periode akhir    : {y.index[-1]}")
print(f"Initial Level    : {L0:.4f}")

display(y.head(10))

INFORMASI DATA
---------------------------------------------
Jumlah observasi : 108
Periode awal     : 1960-01-01 00:00:00
Periode akhir    : 1968-12-01 00:00:00
Initial Level    : 1362.0000


Date/Time
1960-01-01   1,362.0000
1960-02-01     965.0000
1960-03-01     828.0000
1960-04-01     949.0000
1960-05-01     884.0000
1960-06-01     667.0000
1960-07-01   1,415.0000
1960-08-01   6,384.0000
1960-09-01   8,063.0000
1960-10-01   5,413.0000
Name: Nile Discharge, dtype: float64

Visualisasi data aktual menggunakan Plotly.

In [35]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y.index,
    y=y.values,
    mode="lines+markers",
    name="Data Aktual",
    line=dict(width=2),
    marker=dict(size=5)
))

fig.update_layout(
    title="Visualisasi Data Time Series",
    xaxis_title="Periode",
    yaxis_title="Nilai Observasi",
    template="plotly_white",
    hovermode="x unified",
    height=450
)

fig.show()

**Interpretasi:** Amati pola data untuk memahami karakteristik time series sebelum melakukan pemulusan.

### Step 2. Perhitungan SES secara manual

Tujuan: Memahami proses pembaruan level menggunakan parameter smoothing \(\alpha\).

In [36]:
def ses(y, alpha, l0=None):

    y = np.asarray(y, dtype=float)

    if not 0 < alpha <= 1:
        raise ValueError("Alpha harus berada pada interval (0, 1].")

    # Menentukan initial level
    L = y[0] if l0 is None else float(l0)

    forecast = np.zeros(len(y))
    level = np.zeros(len(y))

    for t in range(len(y)):

        # Forecast sebelum observasi diketahui
        forecast[t] = L

        # Memperbarui level
        L = alpha * y[t] + (1 - alpha) * L

        level[t] = L

    return level, forecast

Lakukan simulasi dengan $\alpha=0.3$.

In [37]:
alpha_example = 0.3

level_example, forecast_example = ses(
    y,
    alpha_example,
    L0
)

hasil_ses = pd.DataFrame({
    "Aktual": y.values,
    "Forecast": forecast_example,
    "Updated Level": level_example
}, index=y.index)

display(hasil_ses.head(10))

,Aktual,Forecast,Updated Level
Date/Time,,,
1960-01-01,"1,362.0000","1,362.0000","1,362.0000"
1960-02-01,965.0000,"1,362.0000","1,242.9000"
1960-03-01,828.0000,"1,242.9000","1,118.4300"
1960-04-01,949.0000,"1,118.4300","1,067.6010"
1960-05-01,884.0000,"1,067.6010","1,012.5207"
1960-06-01,667.0000,"1,012.5207",908.8645
1960-07-01,"1,415.0000",908.8645,"1,060.7051"
1960-08-01,"6,384.0000","1,060.7051","2,657.6936"
1960-09-01,"8,063.0000","2,657.6936","4,279.2855"


**Interpretasi:** Semakin besar alpha, semakin besar bobot yang diberikan pada observasi terbaru.

### Step 3. Perhitungan error dan SSE

Tujuan: Mengukur kesalahan forecast untuk mengevaluasi hasil SES.

In [38]:
# Menghitung error
error = y.values - forecast_example

# Menghitung squared error
squared_error = error ** 2

# Membuat tabel evaluasi
hasil_error = pd.DataFrame({
    "Aktual": y.values,
    "Forecast": forecast_example,
    "Error": error,
    "Squared Error": squared_error
}, index=y.index)

# SSE tidak memasukkan observasi pertama
SSE = np.sum(squared_error[1:])

print(f"SSE (Alpha = {alpha_example}) = {SSE:,.4f}")

display(hasil_error.head(10))

SSE (Alpha = 0.3) = 654,503,853.6073


,Aktual,Forecast,Error,Squared Error
Date/Time,,,,
1960-01-01,"1,362.0000","1,362.0000",0.0000,0.0000
1960-02-01,965.0000,"1,362.0000",-397.0000,"157,609.0000"
1960-03-01,828.0000,"1,242.9000",-414.9000,"172,142.0100"
1960-04-01,949.0000,"1,118.4300",-169.4300,"28,706.5249"
1960-05-01,884.0000,"1,067.6010",-183.6010,"33,709.3272"
1960-06-01,667.0000,"1,012.5207",-345.5207,"119,384.5541"
1960-07-01,"1,415.0000",908.8645,506.1355,"256,173.1545"
1960-08-01,"6,384.0000","1,060.7051","5,323.2949","28,337,468.1346"
1960-09-01,"8,063.0000","2,657.6936","5,405.3064","29,217,337.2768"


Buat fungsi SSE agar dapat digunakan kembali:

In [39]:
def sse_ses(alpha, y, l0=None):

    _, forecast = ses(y, alpha, l0)

    errors = np.asarray(y)[1:] - forecast[1:]

    return np.sum(errors ** 2)

### Step 4. Optimasi parameter alpha menggunakan SciPy

Tujuan: Menemukan nilai alpha yang menghasilkan SSE minimum.

Secara matematis:
$$
\alpha^*=\arg\min_{\alpha} SSE(\alpha)
$$

Gunakan minimize_scalar dengan metode bounded.

In [40]:
# Optimasi parameter alpha
hasil_optimasi = minimize_scalar(
    sse_ses,
    args=(y.values, L0),
    bounds=(0.000001, 1.0),
    method="bounded",
    options={"xatol": 1e-10}
)

# Alpha hasil optimasi
alpha_opt = hasil_optimasi.x

# SSE hasil optimasi
sse_opt = sse_ses(alpha_opt, y.values, L0)

# Memeriksa batas alpha = 1 secara eksplisit
sse_boundary = sse_ses(1.0, y.values, L0)

if sse_boundary < sse_opt:
    alpha_opt = 1.0
    sse_opt = sse_boundary

print("HASIL OPTIMASI")
print("-" * 45)
print(f"Alpha optimal : {alpha_opt:.6f}")
print(f"SSE minimum   : {sse_opt:,.4f}")
print(f"Status        : {hasil_optimasi.success}")

HASIL OPTIMASI
---------------------------------------------
Alpha optimal : 1.000000
SSE minimum   : 393,715,723.0000
Status        : True


**Interpretasi:** SciPy mencari nilai alpha yang meminimalkan SSE. Pemeriksaan alpha = 1 dilakukan secara eksplisit karena optimasi bounded tidak menjamin pemeriksaan tepat pada batas atas.

### Step 5. Perbandingan SSE untuk beberapa alpha

Tujuan: Memahami bagaimana perubahan alpha memengaruhi kesalahan model. Hitung SSE untuk alpha 0.1 sampai 1.0.

In [41]:
alpha_values = np.arange(0.1, 1.01, 0.1)

hasil_perbandingan = []

for alpha in alpha_values:

    nilai_sse = sse_ses(
        alpha,
        y.values,
        L0
    )

    hasil_perbandingan.append({
        "Alpha": alpha,
        "SSE": nilai_sse
    })

tabel_sse = pd.DataFrame(hasil_perbandingan)

display(tabel_sse)

,Alpha,SSE
0,0.1000,"676,589,376.6398"
1,0.2000,"677,177,559.7317"
2,0.3000,"654,503,853.6073"
3,0.4000,"614,529,822.8793"
4,0.5000,"568,687,737.2712"
5,0.6000,"524,227,530.2162"
6,0.7000,"484,086,418.7652"
7,0.8000,"448,971,625.9942"
8,0.9000,"418,869,537.1972"
9,1.0000,"393,715,723.0000"


Visualisasi menggunakan Plotly:

In [42]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=tabel_sse["Alpha"],
    y=tabel_sse["SSE"],
    mode="lines+markers",
    name="SSE",
    marker=dict(size=9),
    line=dict(width=2),
    hovertemplate="Alpha: %{x:.2f}<br>SSE: %{y:,.3f}<extra></extra>"
))

fig.add_trace(go.Scatter(
    x=[alpha_opt],
    y=[sse_opt],
    mode="markers",
    name="Alpha Optimal",
    marker=dict(size=14, symbol="star"),
    hovertemplate="Alpha: %{x:.6f}<br>SSE: %{y:,.3f}<extra></extra>"
))

fig.add_vline(
    x=alpha_opt,
    line_dash="dash",
    annotation_text=f"Optimal α = {alpha_opt:.4f}"
)

fig.update_layout(
    title="Perbandingan SSE untuk Berbagai Nilai Alpha",
    xaxis_title="Alpha",
    yaxis_title="Sum of Squared Errors",
    template="plotly_white",
    hovermode="x unified",
    height=500
)

fig.show()

### Step 6. Estimasi SES menggunakan Statsmodels

Tujuan: Melakukan estimasi parameter SES menggunakan library Statsmodels dan membandingkannya dengan perhitungan manual.

In [43]:
# ============================================================
# STEP 6. ESTIMASI SES MENGGUNAKAN STATSMODELS
# ============================================================

sm_model = SimpleExpSmoothing(
    y,
    initialization_method="known",
    initial_level=L0
)

# Optimasi parameter alpha
sm_ses = sm_model.fit(
    optimized=True,
    method="least_squares"
)

# Mengambil parameter alpha
alpha_sm = sm_ses.params["smoothing_level"]

# Menghitung SSE
sse_sm = sse_ses(
    alpha_sm,
    y.values,
    L0
)

# Mengambil final level menggunakan iloc
final_level_sm = sm_ses.level.iloc[-1]

# Forecast satu periode berikutnya
forecast_next_sm = sm_ses.forecast(1).iloc[0]

print("HASIL STATSMODELS")
print("-" * 45)
print(f"Alpha optimal : {alpha_sm:.6f}")
print(f"SSE           : {sse_sm:,.4f}")
print(f"Final Level   : {final_level_sm:,.4f}")
print(f"Forecast      : {forecast_next_sm:,.4f}")

HASIL STATSMODELS
---------------------------------------------
Alpha optimal : 1.000000
SSE           : 393,715,726.3765
Final Level   : 1,209.0000
Forecast      : 1,209.0000


### Step 7. Perbandingan hasil manual dan Statsmodels

Tujuan: Memvalidasi apakah implementasi SES manual menghasilkan nilai yang konsisten dengan Statsmodels.

In [44]:
# Hasil SES manual
level_manual, forecast_manual = ses(
    y,
    alpha_opt,
    L0
)

final_level_manual = level_manual[-1]
forecast_next_manual = final_level_manual

# Hasil Statsmodels
final_level_sm = sm_ses.level.iloc[-1]
forecast_next_sm = sm_ses.forecast(1).iloc[0]

# Tabel perbandingan
perbandingan = pd.DataFrame({
    "Manual": [
        alpha_opt,
        sse_opt,
        final_level_manual,
        forecast_next_manual
    ],
    "Statsmodels": [
        alpha_sm,
        sse_sm,
        final_level_sm,
        forecast_next_sm
    ]
}, index=[
    "Alpha Optimal",
    "SSE",
    "Final Level",
    "Forecast Berikutnya"
])

perbandingan["Selisih Absolut"] = (
    perbandingan["Manual"] -
    perbandingan["Statsmodels"]
).abs()

display(perbandingan)

,Manual,Statsmodels,Selisih Absolut
Alpha Optimal,1.0000,1.0000,0.0000
SSE,"393,715,723.0000","393,715,726.3765",3.3765
Final Level,"1,209.0000","1,209.0000",0.0000
Forecast Berikutnya,"1,209.0000","1,209.0000",0.0000


### Step 8. Visualisasi aktual, smoothing, dan forecast menggunakan Plotly

Tujuan: Membandingkan data aktual, hasil smoothing, dan forecast secara visual.

#### 8.1 Aktual vs hasil smoothing

In [45]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y.index,
    y=y.values,
    mode="lines+markers",
    name="Data Aktual",
    line=dict(width=2),
    marker=dict(size=5)
))

fig.add_trace(go.Scatter(
    x=y.index,
    y=level_manual,
    mode="lines",
    name="SES Smoothed",
    line=dict(width=2, dash="dash")
))

fig.update_layout(
    title=f"SES: Data Aktual vs Smoothed (Alpha = {alpha_opt:.4f})",
    xaxis_title="Periode",
    yaxis_title="Nilai",
    template="plotly_white",
    hovermode="x unified",
    height=500
)

fig.show()

#### 8.2 Perbandingan forecast manual dan Statsmodels

In [46]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y.index,
    y=y.values,
    mode="lines",
    name="Data Aktual"
))

fig.add_trace(go.Scatter(
    x=y.index,
    y=forecast_manual,
    mode="lines",
    name="Forecast Manual",
    line=dict(dash="dash")
))

fig.add_trace(go.Scatter(
    x=y.index,
    y=np.asarray(sm_ses.fittedvalues),
    mode="lines",
    name="Forecast Statsmodels",
    line=dict(dash="dot")
))

fig.update_layout(
    title="Perbandingan Fitted Values: Manual vs Statsmodels",
    xaxis_title="Periode",
    yaxis_title="Nilai",
    template="plotly_white",
    hovermode="x unified",
    height=500
)

fig.show()

#### 8.3 Forecast satu periode berikutnya

In [47]:
# Menentukan indeks periode berikutnya
if isinstance(y.index, pd.DatetimeIndex):

    freq = pd.infer_freq(y.index)

    if freq is not None:
        next_index = pd.date_range(
            start=y.index[-1],
            periods=2,
            freq=freq
        )[1]
    else:
        next_index = (
            y.index[-1] +
            (y.index[-1] - y.index[-2])
        )

elif isinstance(y.index, pd.PeriodIndex):

    next_index = y.index[-1] + 1

else:

    next_index = len(y)

# Membuat grafik forecast
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y.index,
    y=y.values,
    mode="lines+markers",
    name="Data Aktual"
))

fig.add_trace(go.Scatter(
    x=y.index,
    y=level_manual,
    mode="lines",
    name="SES Smoothed",
    line=dict(dash="dash")
))

fig.add_trace(go.Scatter(
    x=[y.index[-1], next_index],
    y=[final_level_manual, forecast_next_manual],
    mode="lines+markers",
    name="Forecast Berikutnya",
    line=dict(width=3, dash="dot"),
    marker=dict(size=10),
    hovertemplate="Periode: %{x}<br>Forecast: %{y:,.3f}<extra></extra>"
))

fig.update_layout(
    title="SES: Forecast Periode Berikutnya",
    xaxis_title="Periode",
    yaxis_title="Nilai",
    template="plotly_white",
    hovermode="x unified",
    height=500
)

fig.show()

print(f"Forecast periode berikutnya: {forecast_next_manual:,.4f}")

Forecast periode berikutnya: 1,209.0000


---
## 3. Holt: Menangkap Tren

Metode **Holt (Double Exponential Smoothing)** merupakan pengembangan SES yang memperhitungkan dua komponen, yaitu *level* (tingkat data) dan *trend* (arah pergerakan data). Metode ini cocok untuk data yang memiliki tren naik atau turun tanpa pola musiman.

$$
\ell_t = \alpha y_t + (1-\alpha)(\ell_{t-1}+b_{t-1})
$$

$$
b_t = \beta(\ell_t-\ell_{t-1})+(1-\beta)b_{t-1}
$$

Prediksi $h$ periode ke depan:

$$
\hat{y}_{T+h}=\ell_T+h\,b_T
$$

Holt menggunakan parameter $\alpha$ untuk pemulusan level dan $\beta$ untuk pemulusan tren. Keduanya menentukan seberapa besar pengaruh data terbaru terhadap estimasi.

**Holt Damped** merupakan pengembangan Holt yang menambahkan parameter $0<\phi<1$ untuk mengurangi pengaruh tren secara bertahap.

$$
\hat{y}_{T+h} = \ell_T+(\phi+\phi^2+\cdots+\phi^h)b_T
$$

Dengan demikian, Holt mengasumsikan tren berlanjut secara linear, sedangkan Holt Damped mengasumsikan tren akan melandai sehingga lebih konservatif untuk prediksi jangka panjang.

**Contoh Perhitungan Holt:**

Diketahui data penjualan sebagai berikut:

| Bulan | Penjualan |
|---|---:|
| Januari | 100 |
| Februari | 110 |
| Maret | 120 |
| April | 130 |
| Mei | 140 |

Gunakan parameter:
- $\alpha = 0.5$ (pemulusan level)
- $\beta = 0.5$ (pemulusan tren)
- Level awal: $\ell_1 = 100$
- Tren awal: $b_1 = 10$

**1. Menghitung Level**

$$
\ell_t = \alpha y_t+(1-\alpha)(\ell_{t-1}+b_{t-1})
$$

Untuk Februari:

$$
\ell_2=0.5(110)+0.5(100+10)=110
$$

**2. Menghitung Tren**

$$
b_t=\beta(\ell_t-\ell_{t-1})+(1-\beta)b_{t-1}
$$

Untuk Februari:

$$
b_2=0.5(110-100)+0.5(10)=10
$$

**3. Menghitung Prediksi**

$$
\hat{y}_{T+h}=\ell_T+h\,b_T
$$

Prediksi penjualan Juni:

$$
\hat{y}_{6}=140+1(10)=150
$$

**4. Hasil Perhitungan**

| Bulan | Aktual | Level ($\ell_t$) | Tren ($b_t$) |
|---|---:|---:|---:|
| Januari | 100 | 100 | 10 |
| Februari | 110 | 110 | 10 |
| Maret | 120 | 120 | 10 |
| April | 130 | 130 | 10 |
| Mei | 140 | 140 | 10 |
| Juni | - | - | - |

**Kesimpulan:** Metode Holt menghasilkan prediksi penjualan bulan Juni sebesar **150 unit**. Karena data menunjukkan kenaikan konsisten sebesar 10 unit setiap bulan, estimasi tren tetap sebesar 10.

**Ilustrasi Perbandingan SES, Holt, dan Holt Damped**

Skenario: Data PDB riil Amerika Serikat (gdp) digunakan untuk memprediksi 10 periode terakhir. Data dibagi menjadi data latih (training) dan data uji (testing).
- SES: hanya memperhitungkan level, tanpa tren.
- Holt: memperhitungkan level dan tren linear.
- Holt Damped: memperhitungkan level dan tren yang secara bertahap melandai.

### Step 1. Persiapan Data dan Pemodelan

In [48]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from statsmodels.tsa.holtwinters import (
    SimpleExpSmoothing,
    Holt
)

# Membagi data menjadi training dan testing
latih = gdp.iloc[:-10]
uji = gdp.iloc[-10:]

# Membangun tiga model
model = {
    'SES': SimpleExpSmoothing(
        latih,
        initialization_method='estimated'
    ).fit(),

    'Holt': Holt(
        latih,
        initialization_method='estimated'
    ).fit(),

    'Holt Damped': Holt(
        latih,
        damped_trend=True,
        initialization_method='estimated'
    ).fit()
}

print(f"Jumlah data training : {len(latih)}")
print(f"Jumlah data testing  : {len(uji)}")

Jumlah data training : 26
Jumlah data testing  : 10


### Step 2. Menghasilkan Prediksi

Setiap model menghasilkan ramalan untuk 10 periode ke depan. Periode ramalan tersebut kemudian dibandingkan dengan data aktual pada data uji.

In [49]:
# Menghasilkan prediksi dari masing-masing model
prediksi = {
    nama: hasil.forecast(len(uji))
    for nama, hasil in model.items()
}

### Step 3. Visualisasi Perbandingan

Visualisasi berikut memperlihatkan data aktual, periode training, periode testing, dan hasil prediksi ketiga metode.

In [50]:
fig = go.Figure()

# Data aktual
fig.add_trace(go.Scatter(
    x=gdp.index.astype(str),
    y=gdp.values,
    mode='lines',
    name='Data Aktual',
    line=dict(color='black', width=2)
))

# Prediksi masing-masing model
warna = {
    'SES': '#A0AEC0',
    'Holt': '#C53030',
    'Holt Damped': '#2B6CB0'
}

for nama, forecast in prediksi.items():
    fig.add_trace(go.Scatter(
        x=forecast.index.astype(str),
        y=forecast.values,
        mode='lines+markers',
        name=nama,
        line=dict(color=warna[nama], width=2),
        marker=dict(size=6)
    ))

# Batas awal data testing
fig.add_shape(
    type='line',
    x0=str(uji.index[0]),
    x1=str(uji.index[0]),
    y0=0,
    y1=1,
    yref='paper',
    line=dict(color='gray', dash='dot', width=2)
)

fig.add_annotation(
    x=str(uji.index[0]),
    y=1,
    yref='paper',
    text='Awal Testing',
    showarrow=False,
    yshift=12
)

fig.update_layout(
    title='Perbandingan Prediksi SES, Holt, dan Holt Damped',
    xaxis_title='Periode',
    yaxis_title='PDB Riil',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', y=1.12),
    height=500
)

fig.show()

### Step 4. Evaluasi dan Perbandingan Model

Gunakan MAPE untuk mengukur kesalahan prediksi pada data testing. AICc digunakan sebagai informasi tambahan untuk membandingkan kecocokan model pada data training.

In [51]:
# Fungsi menghitung MAPE
def hitung_mape(aktual, prediksi):
    aktual = np.asarray(aktual, dtype=float)
    prediksi = np.asarray(prediksi, dtype=float)

    mask = aktual != 0

    return np.mean(
        np.abs((aktual[mask] - prediksi[mask]) / aktual[mask])
    ) * 100


# Menyusun hasil evaluasi
hasil = []

for nama, fitted in model.items():

    hasil.append({
        'Metode': nama,
        'Alpha': fitted.params.get('smoothing_level', np.nan),
        'Beta': fitted.params.get('smoothing_trend', np.nan),
        'Phi': fitted.params.get('damping_trend', np.nan),
        'MAPE Testing (%)': hitung_mape(
            uji,
            prediksi[nama]
        ),
        'AICc': fitted.aicc
    })

tabel_hasil = pd.DataFrame(hasil).set_index('Metode')

display(tabel_hasil.round(4))

,Alpha,Beta,Phi,MAPE Testing (%),AICc
Metode,,,,,
SES,1.0000,NaN,NaN,4.6139,219.7233
Holt,0.9627,0.2908,NaN,2.9951,198.4801
Holt Damped,0.9447,0.3383,0.9950,2.7678,202.3688


### Step 5. Interpretasi Hasil

Perhatikan tiga hal berikut ketika membaca hasil:

1. Pola prediksi: SES cenderung menghasilkan garis datar, Holt mengikuti tren linear, sedangkan Holt Damped menghasilkan tren yang semakin melandai.
2. MAPE Testing: semakin kecil nilai MAPE, semakin kecil rata-rata kesalahan prediksi pada data pengujian.
3. AICc: semakin kecil nilainya, semakin baik keseimbangan antara kecocokan model dan kompleksitasnya pada data training.

**Kesimpulan pembelajaran:** Tidak semua data cocok menggunakan metode yang sama. Jika data memiliki tren, Holt dapat menjadi pilihan dibandingkan SES. Namun, jika tren diperkirakan melandai, Holt Damped dapat memberikan hasil ramalan yang lebih realistis.

---

## 4. Holt-Winters: Menangkap Tren dan Musiman

Metode **Holt-Winters (Triple Exponential Smoothing)** merupakan pengembangan metode Holt dengan menambahkan komponen musiman (*seasonality*). Metode ini menggunakan tiga komponen, yaitu **level**, **trend**, dan **seasonality**. Cocok digunakan pada data yang memiliki tren sekaligus pola musiman.

Terdapat dua pendekatan dalam Holt-Winters:

**A. Holt-Winters Aditif**

Digunakan ketika amplitudo atau besarnya fluktuasi musiman relatif konstan dari waktu ke waktu.

Persamaan pembaruan level:

$$
\ell_t = \alpha(y_t-s_{t-m})+
(1-\alpha)(\ell_{t-1}+b_{t-1})
$$

Persamaan pembaruan tren:

$$
b_t = \beta(\ell_t-\ell_{t-1})+
(1-\beta)b_{t-1}
$$

Persamaan pembaruan musiman:

$$
s_t = \gamma(y_t-\ell_{t-1}-b_{t-1})+
(1-\gamma)s_{t-m}
$$

Prediksi $h$ periode ke depan:

$$
\hat{y}_{T+h} =
\ell_T+h b_T+s_{T+h-m(k+1)}
$$

**B. Holt-Winters Multiplikatif**

Digunakan ketika amplitudo atau besarnya fluktuasi musiman berubah mengikuti tingkat data. Komponen musiman dinyatakan sebagai faktor pengali, bukan penjumlahan.

Persamaan pembaruan level:

$$
\ell_t = \alpha\left(\frac{y_t}{s_{t-m}}\right)+
(1-\alpha)(\ell_{t-1}+b_{t-1})
$$

Persamaan pembaruan tren:

$$
b_t = \beta(\ell_t-\ell_{t-1})+
(1-\beta)b_{t-1}
$$

Persamaan pembaruan musiman:

$$
s_t = \gamma\left(\frac{y_t}{\ell_{t-1}+b_{t-1}}\right)+
(1-\gamma)s_{t-m}
$$

Prediksi $h$ periode ke depan:

$$
\hat{y}_{T+h} =
(\ell_T+h b_T)s_{T+h-m(k+1)}
$$

**Keterangan Parameter**

- $y_t$: nilai aktual pada periode ke-$t$.
- $\ell_t$: estimasi level.
- $b_t$: estimasi tren.
- $s_t$: estimasi komponen musiman.
- $m$: panjang satu siklus musiman.
- $\alpha$: parameter pemulusan level.
- $\beta$: parameter pemulusan tren.
- $\gamma$: parameter pemulusan musiman.
- $k$: jumlah siklus musiman yang telah dilewati dalam horizon prediksi.

Ketiga parameter pemulusan mengendalikan seberapa besar pengaruh informasi terbaru terhadap estimasi masing-masing komponen.

**Perbedaan utama:**
- **Aditif:** pola musiman memiliki besaran relatif tetap.
- **Multiplikatif:** pola musiman berubah sebanding dengan tingkat data.

Berikut kode untuk membandingkan Holt-Winters Aditif dan Multiplikatif sekaligus, baik menggunakan implementasi manual maupun Statsmodels.

### Step 1. Persiapan Library dan Data

Memuat library yang diperlukan dan membagi dataset car menjadi data training dan testing. Dua belas observasi terakhir digunakan sebagai data testing untuk mengevaluasi kemampuan peramalan model.

In [52]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from scipy.optimize import minimize
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Panjang periode musiman
m = 12

# Membagi data training dan testing
latih = car.iloc[:-12].astype(float)
uji = car.iloc[-12:].astype(float)

print("Jumlah data training:", len(latih))
print("Jumlah data testing :", len(uji))

# Visualisasi data
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=latih.index,
    y=latih.values,
    mode='lines',
    name='Training'
))

fig.add_trace(go.Scatter(
    x=uji.index,
    y=uji.values,
    mode='lines',
    name='Testing'
))

fig.update_layout(
    title='Pembagian Data Training dan Testing',
    xaxis_title='Periode',
    yaxis_title='Penjualan',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

Jumlah data training: 96
Jumlah data testing : 12


**Penjelasan:** Data training digunakan untuk membangun model dan mengestimasi parameter. Data testing digunakan untuk mengukur akurasi ramalan pada data yang belum digunakan dalam proses pelatihan.

### Step 2. Membuat Kelas Holt-Winters Manual

Pada tahap ini, kita membangun satu kelas Python yang dapat menjalankan dua pendekatan sekaligus, yaitu aditif dan multiplikatif.

Komponen yang dihitung meliputi:
- Level $\ell_t$: estimasi tingkat data.
- Tren $b_t$: estimasi arah pergerakan data.
- Musiman $s_t$: estimasi pola berulang setiap 12 periode.

In [53]:
class HoltWinters:

    def __init__(self, m=12, seasonal='add'):
        self.m = m
        self.seasonal = seasonal

    def _inisialisasi(self, y):

        m = self.m

        level = np.mean(y[:m])

        trend = (
            np.mean(y[m:2*m]) - np.mean(y[:m])
        ) / m

        if self.seasonal == 'add':
            seasonal = y[:m] - level
        else:
            seasonal = y[:m] / level

        return level, trend, list(seasonal)

    def _jalankan(self, y, alpha, beta, gamma):

        m = self.m

        level, trend, seasonal = self._inisialisasi(y)

        fitted = np.empty(len(y))

        for t, aktual in enumerate(y):

            s = seasonal[t]

            # Ramalan satu langkah ke depan
            if self.seasonal == 'add':
                fitted[t] = level + trend + s
            else:
                fitted[t] = (level + trend) * s

            # Memperbarui level
            if self.seasonal == 'add':
                level_baru = (
                    alpha * (aktual - s)
                    + (1-alpha) * (level + trend)
                )
            else:
                level_baru = (
                    alpha * (aktual / s)
                    + (1-alpha) * (level + trend)
                )

            # Memperbarui musiman
            if self.seasonal == 'add':
                seasonal_baru = (
                    gamma * (aktual - level - trend)
                    + (1-gamma) * s
                )
            else:
                seasonal_baru = (
                    gamma * (aktual / (level + trend))
                    + (1-gamma) * s
                )

            # Memperbarui tren
            trend = (
                beta * (level_baru - level)
                + (1-beta) * trend
            )

            level = level_baru
            seasonal.append(seasonal_baru)

        return fitted, (level, trend, seasonal[-m:])

    def fit(self, y):

        y = np.asarray(y, dtype=float)

        if self.m < 2 or len(y) < 2*self.m:
            raise ValueError("Minimal diperlukan dua musim penuh.")

        if not np.all(np.isfinite(y)):
            raise ValueError("Data mengandung NaN atau inf.")

        if self.seasonal == 'mul' and np.any(y <= 0):
            raise ValueError("Model multiplikatif membutuhkan data positif.")

        # Fungsi objektif SSE
        def objektif(params):

            fitted, _ = self._jalankan(y, *params)

            error = y[self.m:] - fitted[self.m:]

            return np.sum(error**2)

        # Optimasi alpha, beta, dan gamma
        hasil = minimize(
            objektif,
            x0=[0.3, 0.05, 0.1],
            bounds=[(1e-4, 1.0)] * 3,
            method='L-BFGS-B'
        )

        if not hasil.success:
            raise RuntimeError(f"Optimasi gagal: {hasil.message}")

        self.params_ = dict(zip(
            ['alpha', 'beta', 'gamma'],
            hasil.x
        ))

        self.fitted_, self.state_ = self._jalankan(
            y, *hasil.x
        )

        self.sse_ = hasil.fun

        return self

    def forecast(self, h):

        if not hasattr(self, 'state_'):
            raise RuntimeError("Model harus di-fit terlebih dahulu.")

        if h < 1:
            raise ValueError("Horizon forecast minimal 1.")

        level, trend, seasonal = self.state_

        hasil = []

        for k in range(h):

            nilai_tren = level + (k+1)*trend
            s = seasonal[k % self.m]

            if self.seasonal == 'add':
                hasil.append(nilai_tren + s)
            else:
                hasil.append(nilai_tren * s)

        return np.array(hasil)

### Step 3. Membangun Model Holt-Winters Aditif

Model aditif menggunakan komponen musiman dalam bentuk penjumlahan. Pendekatan ini sesuai ketika fluktuasi musiman relatif konstan meskipun level data berubah.

In [54]:
# Membuat model aditif manual
hw_aditif = HoltWinters(
    m=12,
    seasonal='add'
).fit(latih)

# Menampilkan parameter
print("Parameter Holt-Winters Aditif")

for nama, nilai in hw_aditif.params_.items():
    print(f"{nama}: {nilai:.4f}")

print(f"SSE Training: {hw_aditif.sse_:.4f}")

Parameter Holt-Winters Aditif
alpha: 0.1294
beta: 0.0001
gamma: 0.4254
SSE Training: 204204469.3563


### Step 4. Membangun Model Holt-Winters Multiplikatif

Model multiplikatif menggunakan komponen musiman sebagai faktor pengali. Pendekatan ini sesuai ketika amplitudo fluktuasi musiman berubah mengikuti level data.

In [55]:
# Membuat model multiplikatif manual
hw_multiplikatif = HoltWinters(
    m=12,
    seasonal='mul'
).fit(latih)

# Menampilkan parameter
print("Parameter Holt-Winters Multiplikatif")

for nama, nilai in hw_multiplikatif.params_.items():
    print(f"{nama}: {nilai:.4f}")

print(f"SSE Training: {hw_multiplikatif.sse_:.4f}")

Parameter Holt-Winters Multiplikatif
alpha: 0.2848
beta: 0.0001
gamma: 0.2045
SSE Training: 196642672.9887


### Step 5. Estimasi Menggunakan Statsmodels

Selanjutnya, kedua pendekatan diimplementasikan menggunakan library Statsmodels sebagai pembanding hasil perhitungan manual.

In [56]:
# Statsmodels Aditif
hw_aditif_sm = ExponentialSmoothing(
    latih,
    trend='add',
    seasonal='add',
    seasonal_periods=12,
    initialization_method='estimated'
).fit()

# Statsmodels Multiplikatif
hw_multiplikatif_sm = ExponentialSmoothing(
    latih,
    trend='add',
    seasonal='mul',
    seasonal_periods=12,
    initialization_method='estimated'
).fit()

print("Model Statsmodels berhasil diestimasi.")

Model Statsmodels berhasil diestimasi.


### Step 6. Menghasilkan Prediksi Empat Model

Keempat model digunakan untuk memprediksi 12 periode ke depan. Hasil prediksi kemudian disimpan dalam dictionary agar mudah dibandingkan.

In [57]:
# Daftar model
model = {
    'Aditif Manual': hw_aditif,
    'Aditif Statsmodels': hw_aditif_sm,
    'Multiplikatif Manual': hw_multiplikatif,
    'Multiplikatif Statsmodels': hw_multiplikatif_sm
}

# Menghasilkan prediksi
prediksi = {
    nama: hasil.forecast(len(uji))
    for nama, hasil in model.items()
}

# Tabel hasil prediksi
tabel_prediksi = pd.DataFrame(
    prediksi,
    index=uji.index
)

tabel_prediksi.insert(0, 'Aktual', uji)

display(tabel_prediksi.round(2))

,Aktual,Aditif Manual,Aditif Statsmodels,Multiplikatif Manual,Multiplikatif Statsmodels
Month,,,,,
1968-01-01,"13,210.0000","12,848.5700","14,109.0400","12,716.9000","13,212.5400"
1968-02-01,"14,251.0000","12,966.5600","14,754.6300","13,527.2300","13,831.8400"
1968-03-01,"20,139.0000","20,892.5100","20,263.9500","20,838.1200","20,957.8400"
1968-04-01,"21,725.0000","21,744.1800","22,505.9900","23,186.6200","23,548.2300"
1968-05-01,"26,099.0000","23,656.0700","23,749.9700","24,851.2400","25,033.6000"
1968-06-01,"21,084.0000","21,711.1600","21,472.2400","22,533.8800","22,270.8400"
1968-07-01,"18,024.0000","15,530.8900","16,632.0800","16,230.1600","16,288.0300"
1968-08-01,"16,722.0000","14,461.6500","14,448.5800","14,087.4800","13,769.7800"
1968-09-01,"14,385.0000","13,421.0200","13,119.8800","12,368.5400","12,253.5100"


### Step 7. Menghitung MAPE dan SSE

MAPE digunakan untuk mengukur kesalahan prediksi pada data testing. Sementara itu, SSE menunjukkan jumlah kuadrat kesalahan pada data training.

In [58]:
# Fungsi menghitung MAPE
def mape(aktual, ramalan):

    aktual = np.asarray(aktual, dtype=float)
    ramalan = np.asarray(ramalan, dtype=float)

    mask = aktual != 0

    return np.mean(
        np.abs(
            (aktual[mask] - ramalan[mask]) / aktual[mask]
        )
    ) * 100


hasil_evaluasi = []

for nama, fitted in model.items():

    if hasattr(fitted, 'params_'):

        params = fitted.params_
        sse = fitted.sse_

        alpha = params['alpha']
        beta = params['beta']
        gamma = params['gamma']

    else:

        params = fitted.params
        sse = fitted.sse

        alpha = params['smoothing_level']
        beta = params['smoothing_trend']
        gamma = params['smoothing_seasonal']

    hasil_evaluasi.append({
        'Model': nama,
        'Alpha': alpha,
        'Beta': beta,
        'Gamma': gamma,
        'SSE Training': sse,
        'MAPE Testing (%)': mape(uji, prediksi[nama])
    })

tabel_hasil = pd.DataFrame(
    hasil_evaluasi
).set_index('Model')

display(tabel_hasil.round(4))

,Alpha,Beta,Gamma,SSE Training,MAPE Testing (%)
Model,,,,,
Aditif Manual,0.1294,0.0001,0.4254,"204,204,469.3563",7.0178
Aditif Statsmodels,0.1685,0.0000,0.0000,"161,339,816.2773",7.2107
Multiplikatif Manual,0.2848,0.0001,0.2045,"196,642,672.9887",7.7628
Multiplikatif Statsmodels,0.3019,0.0000,0.0000,"158,495,579.9282",7.9745


### Step 8. Visualisasi Perbandingan Empat Model

Visualisasi interaktif digunakan untuk membandingkan data aktual dengan hasil prediksi keempat model.

In [59]:
# ============================================
# VISUALISASI PERBANDINGAN 4 MODEL HOLT-WINTERS
# ============================================

import plotly.graph_objects as go

# Membuat figure
fig = go.Figure()

# --------------------------------------------
# 1. Data aktual
# --------------------------------------------

fig.add_trace(go.Scatter(
    x=car.index,
    y=car.values,
    mode='lines',
    name='Data Aktual',
    line=dict(
        color='black',
        width=2
    )
))

# --------------------------------------------
# 2. Warna masing-masing model
# --------------------------------------------

warna = {
    'Aditif Manual': '#C53030',
    'Aditif Statsmodels': '#FC8181',
    'Multiplikatif Manual': '#2B6CB0',
    'Multiplikatif Statsmodels': '#63B3ED'
}

# --------------------------------------------
# 3. Menampilkan hasil prediksi
# --------------------------------------------

for nama, ramalan in prediksi.items():

    fig.add_trace(go.Scatter(
        x=uji.index,
        y=ramalan,
        mode='lines+markers',
        name=nama,
        line=dict(
            color=warna[nama],
            width=2
        ),
        marker=dict(size=6)
    ))

# --------------------------------------------
# 4. Garis pemisah Training dan Testing
# --------------------------------------------

tanggal_awal_testing = str(uji.index[0])

fig.add_shape(
    type='line',
    x0=tanggal_awal_testing,
    x1=tanggal_awal_testing,
    y0=0,
    y1=1,
    xref='x',
    yref='paper',
    line=dict(
        color='gray',
        width=2,
        dash='dot'
    )
)

# --------------------------------------------
# 5. Keterangan garis pemisah
# --------------------------------------------

fig.add_annotation(
    x=tanggal_awal_testing,
    y=1,
    xref='x',
    yref='paper',
    text='Awal Testing',
    showarrow=False,
    yshift=12,
    font=dict(
        color='gray',
        size=11
    )
)

# --------------------------------------------
# 6. Pengaturan layout
# --------------------------------------------

fig.update_layout(
    title=dict(
        text='Perbandingan Holt-Winters Aditif dan Multiplikatif',
        x=0.5
    ),
    xaxis_title='Periode',
    yaxis_title='Penjualan',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        orientation='h',
        y=-0.2,
        x=0.5,
        xanchor='center'
    ),
    height=550
)

# --------------------------------------------
# 7. Menampilkan visualisasi
# --------------------------------------------

fig.show()

### Step 9. Menentukan Model Terbaik

Model terbaik ditentukan berdasarkan nilai MAPE terkecil pada data testing.

In [60]:
model_terbaik = tabel_hasil[
    'MAPE Testing (%)'
].idxmin()

nilai_mape = tabel_hasil.loc[
    model_terbaik,
    'MAPE Testing (%)'
]

print("Model terbaik :", model_terbaik)
print(f"MAPE Testing  : {nilai_mape:.2f}%")

Model terbaik : Aditif Manual
MAPE Testing  : 7.02%


## 5. Panduan Memilih Metode Pemulusan

Pemilihan metode pemulusan (*Exponential Smoothing*) harus disesuaikan dengan pola data *time series*, khususnya keberadaan tren dan pola musiman.

| Karakteristik Pola Data | Metode yang Direkomendasikan |
|---|---|
| Tanpa tren dan tanpa musiman | Simple Exponential Smoothing (SES) |
| Memiliki tren, tanpa musiman | Holt (Damped Holt untuk peramalan jangka panjang) |
| Memiliki tren dan musiman dengan amplitudo relatif konstan | Holt-Winters Aditif |
| Memiliki tren dan musiman dengan amplitudo sebanding terhadap level data | Holt-Winters Multiplikatif |

**Hal yang Perlu Diperhatikan**

1. **Sesuaikan metode dengan pola data.** Perhatikan keberadaan tren, musiman, dan perubahan amplitudo musiman.
2. **Gunakan tolok ukur (*benchmark*) sederhana.** Bandingkan hasil model dengan metode Naive dan Seasonal Naive. Model yang lebih kompleks harus mampu memberikan hasil lebih baik dibandingkan tolok ukur tersebut.
3. **Jangan hanya mengandalkan satu periode pengujian.** Selisih kecil antarmetode pada data testing belum tentu menunjukkan keunggulan yang konsisten.
4. **Gunakan validasi silang deret waktu (*Time Series Cross-Validation*).** Evaluasi melalui beberapa periode training dan testing untuk memperoleh penilaian kinerja model yang lebih stabil.

**Kesimpulan:** Metode terbaik bukanlah metode yang paling kompleks, melainkan metode yang sesuai dengan karakteristik data dan mampu menghasilkan akurasi lebih baik dibandingkan metode sederhana secara konsisten.

---

## 6. Interval Prediksi dengan ETS

`ETSModel` (*Error-Trend-Seasonal*) memformulasikan metode *Exponential Smoothing* dalam kerangka model statistik **state-space**. Pendekatan ini memungkinkan estimasi komponen error, tren, dan musiman secara lebih eksplisit.

Selain menghasilkan ramalan titik (*point forecast*), model ETS juga dapat digunakan untuk memperoleh:

- **Interval prediksi (*Prediction Interval*):** rentang nilai yang diperkirakan akan mencakup observasi aktual pada tingkat kepercayaan tertentu.
- **Likelihood:** ukuran kecocokan model terhadap data berdasarkan probabilitas yang diestimasi.
- **AIC (*Akaike Information Criterion*):** kriteria untuk membandingkan model dengan mempertimbangkan kecocokan dan kompleksitas model. Nilai AIC yang lebih kecil umumnya menunjukkan model yang lebih baik di antara model yang dibandingkan.

Dengan demikian, ETS tidak hanya digunakan untuk menghasilkan ramalan, tetapi juga menyediakan dasar statistik untuk mengevaluasi ketidakpastian prediksi dan membandingkan model.

In [61]:
# ============================================
# 1. ESTIMASI MODEL ETS
# ============================================

from statsmodels.tsa.exponential_smoothing.ets import ETSModel
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Membangun model ETS
ets = ETSModel(
    latih.astype(float),
    error='mul',
    trend='add',
    seasonal='mul',
    seasonal_periods=12
).fit(disp=False)

# ============================================
# 2. MENGHASILKAN INTERVAL PREDIKSI
# ============================================

pr = ets.get_prediction(
    start=uji.index[0],
    end=uji.index[-1]
).summary_frame(alpha=0.05)

# Menghitung cakupan interval prediksi
cakup = (
    (uji >= pr['pi_lower']) &
    (uji <= pr['pi_upper'])
).mean()

# ============================================
# 3. VISUALISASI AKTUAL DAN PREDIKSI
# ============================================

fig = go.Figure()

# Data aktual
data_visual = car.loc['1965':]

fig.add_trace(go.Scatter(
    x=data_visual.index,
    y=data_visual.values,
    mode='lines',
    name='Data Aktual',
    line=dict(color='black', width=1.5)
))

# Batas atas interval prediksi
fig.add_trace(go.Scatter(
    x=pr.index,
    y=pr['pi_upper'],
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

# Batas bawah dan area interval prediksi
fig.add_trace(go.Scatter(
    x=pr.index,
    y=pr['pi_lower'],
    mode='lines',
    line=dict(width=0),
    fill='tonexty',
    fillcolor='rgba(197, 48, 48, 0.15)',
    name='Interval Prediksi 95%',
    hovertemplate='Batas bawah: %{y:.2f}<extra></extra>'
))

# Ramalan titik
fig.add_trace(go.Scatter(
    x=pr.index,
    y=pr['mean'],
    mode='lines+markers',
    name='ETS(M,A,M)',
    line=dict(color='#C53030', width=2),
    marker=dict(size=5)
))

# Garis awal periode testing
fig.add_shape(
    type='line',
    x0=str(uji.index[0]),
    x1=str(uji.index[0]),
    y0=0,
    y1=1,
    xref='x',
    yref='paper',
    line=dict(color='gray', dash='dot', width=1.5)
)

fig.update_layout(
    title='ETS(M,A,M): Ramalan dan Interval Prediksi 95%',
    xaxis_title='Periode',
    yaxis_title='Nilai Aktual',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', y=-0.2),
    height=500
)

fig.show()

# ============================================
# 4. EVALUASI MODEL
# ============================================

print('Evaluasi Model ETS(M,A,M)')
print('-' * 45)

print(f'AICc                 : {ets.aicc:.2f}')
print(f'MAPE Testing         : {mape(uji, pr["mean"]):.2f}%')
print(f'Cakupan Interval 95% : {cakup:.0%}')

Evaluasi Model ETS(M,A,M)
---------------------------------------------
AICc                 : 1710.69
MAPE Testing         : 8.16%
Cakupan Interval 95% : 100%


> Semua 12 nilai uji berada di dalam interval 95% — konsisten dengan interval yang terkalibrasi, meski 12 titik terlalu
> sedikit untuk memastikannya. AICc ETS **tidak dapat dibandingkan** langsung dengan AICc `ExponentialSmoothing` di atas:
> keduanya memakai rumusan likelihood yang berbeda. Bandingkan AIC hanya antar-model dengan rumusan dan data yang sama.

---
## 7. Latihan

### LATIHAN 1 — Tugas 2, langkah 1
Terapkan minimal dua metode pemulusan (mis. HW aditif dan multiplikatif) pada dataset Tugas Anda. Sisihkan periode terakhir
sebagai data uji; laporkan MAE, RMSE, MAPE, dan bandingkan dengan seasonal naive.

In [62]:
# TODO

### LATIHAN 2 — Lengkapi kelas
Tambahkan metode `forecast_interval(h, n_sim=1000)` ke `HoltWintersAditif` yang membuat interval 95% dengan **simulasi**:
tambahkan galat acak yang diambil ulang (*bootstrap*) dari residu latih ke rekursi masa depan.

In [63]:
# TODO

### LATIHAN 3 — Holt pada log
Bandingkan Holt pada PDB asli dan Holt pada log PDB (ramalan di-`exp`-kan kembali). Mana yang lebih masuk akal untuk deret
yang tumbuh dengan persentase relatif konstan?

In [64]:
# TODO

---
## 8. Ringkasan Pertemuan 5

- ▸ Pemulusan eksponensial = rata-rata terboboti dengan bobot menurun eksponensial; α mengatur kecepatan adaptasi.
- ▸ SES (level), Holt (+tren, opsi damped), Holt-Winters (+musiman aditif/multiplikatif).
- ▸ Parameter dipilih dengan meminimalkan galat ramalan satu langkah (atau likelihood/AICc).
- ▸ Pilih metode sesuai pola data dan buktikan dengan data uji kronologis dibanding tolok ukur seasonal naive.
- ▸ ETS memberi interval prediksi yang melebar seiring horizon.

### Tugas mandiri (PT & BM)
Baca Hyndman & Athanasopoulos (2021) Bab 8.

**Pertemuan berikutnya:** `P06_Identifikasi_ARIMA_SARIMA.ipynb`